In [3]:
import pdfplumber

def group_words_into_lines(words, y_tolerance=2.0):
    """
    Group words into lines by their top position, within a y_tolerance.
    """
    lines = []
    
    # Sort words top to bottom, then left to right
    words_sorted = sorted(words, key=lambda w: (w['top'], w['x0']))

    current_line = []
    current_top = None

    for word in words_sorted:
        word_top = word['top']
        
        if current_line == []:
            # Start a new line
            current_line = [word]
            current_top = word_top
        else:
            if abs(word_top - current_top) <= y_tolerance:
                current_line.append(word)
            else:
                # Finish the current line
                lines.append(current_line)
                # Start a new line
                current_line = [word]
                current_top = word_top
    
    # Add last line
    if current_line:
        lines.append(current_line)

    return lines

# Open the PDF
with pdfplumber.open(r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

    # Group words into lines
    lines = group_words_into_lines(words)

    # Output line info
    for line_words in lines:
        x0 = min(w['x0'] for w in line_words)
        x1 = max(w['x1'] for w in line_words)
        top = min(w['top'] for w in line_words)
        bottom = max(w['bottom'] for w in line_words)
        text = " ".join(w['text'] for w in line_words)
        print(f"Line: '{text}' | x0: {x0:.2f} | top: {top:.2f} | x1: {x1:.2f} | bottom: {bottom:.2f}")

Line: 'Bill To : CW Management Pty Ltd B655' | x0: 27.84 | top: 114.57 | x1: 235.61 | bottom: 125.10
Line: 'Customer No. : PH1871' | x0: 370.79 | top: 117.45 | x1: 508.61 | bottom: 127.98
Line: 'Invoice No. : 91060' | x0: 370.79 | top: 133.65 | x1: 499.76 | bottom: 144.18
Line: 'Ship To : Chemist Warehouse Townsville B158' | x0: 27.84 | top: 137.49 | x1: 271.91 | bottom: 148.02
Line: 'Invoice Date : 15.08.23' | x0: 370.79 | top: 149.97 | x1: 511.45 | bottom: 160.50
Line: '126-128 Sturt St Address :' | x0: 28.80 | top: 157.65 | x1: 165.02 | bottom: 169.15
Line: 'Due Date : 30.09.23' | x0: 370.79 | top: 166.29 | x1: 511.45 | bottom: 176.82
Line: 'TOWNSVILLE QLD 4810' | x0: 85.32 | top: 171.09 | x1: 206.26 | bottom: 181.63
Line: 'Invoice Amount : $ 187.25 AUD AUSTRALIA' | x0: 85.32 | top: 182.49 | x1: 558.13 | bottom: 194.94
Line: 'TTAAXX IINNVVOOIICCEE' | x0: 265.92 | top: 230.83 | x1: 341.31 | bottom: 245.19
Line: 'PPaaggee 11 ooff 11' | x0: 494.16 | top: 237.61 | x1: 566.24 | bottom: 2

In [4]:
import pdfplumber
from difflib import SequenceMatcher

# Keywords
HEADER_KEYWORDS = ["description", "desc", "qty", "quantity", "item no", "code", "product", "uom", "unit"]
FOOTER_KEYWORDS = ["subtotal", "total", "grand total", "gst", "tax"]

def fuzzy_match(needle, haystack, threshold=0.7):
    """Fuzzy match a keyword against a line, return True if close enough."""
    return SequenceMatcher(None, needle, haystack).ratio() >= threshold

def group_words_to_lines(words, y_tolerance=3):
    """Group words into lines by Y position (with tolerance for slight variation)."""
    lines = {}
    for w in words:
        y_key = round(w["top"] / y_tolerance) * y_tolerance
        if y_key not in lines:
            lines[y_key] = []
        lines[y_key].append(w)
    return lines

# Open PDF
with pdfplumber.open(r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

    # Group words into lines
    grouped_lines = group_words_to_lines(words)

    header_y = None
    footer_y = None

    # Detect header/footer lines
    for y, line_words in sorted(grouped_lines.items()):
        # Rebuild line text
        line_text = " ".join(w["text"] for w in sorted(line_words, key=lambda w: w["x0"]))
        line_text_lower = line_text.lower()
        print(f"[Y={y}] {line_text}")  # Debug: print every reconstructed line

        # Header detection with fuzzy match
        if header_y is None:
            for kw in HEADER_KEYWORDS:
                if fuzzy_match(kw, line_text_lower):
                    header_y = min(w["top"] for w in line_words)
                    print(f"✅ Fuzzy matched table header at Y: {header_y} | Line: {line_text}")
                    break

        # Footer detection (exact match for totals)
        if footer_y is None:
            if any(kw in line_text_lower for kw in FOOTER_KEYWORDS):
                footer_y = min(w["top"] for w in line_words)
                print(f"✅ Found table footer at Y: {footer_y} | Line: {line_text}")

    # Heuristic fallback if header not found
    if header_y is None and footer_y is not None:
        header_y = footer_y - 100
        print(f"⚠️ Header not found. Assuming header Y at {header_y} based on footer.")

    # Segment areas
    top_area = [w for w in words if header_y is not None and w["top"] < header_y]
    line_items_area = [w for w in words if header_y is not None and footer_y is not None and header_y <= w["top"] <= footer_y]
    bottom_area = [w for w in words if footer_y is not None and w["top"] > footer_y]

    # Print segmented areas
    print("\n=== TOP AREA ===")
    for w in top_area:
        print(f"{w['text']} at Y: {w['top']}")

    print("\n=== LINE ITEMS AREA ===")
    for w in line_items_area:
        print(f"{w['text']} at Y: {w['top']}")

    print("\n=== BOTTOM AREA ===")
    for w in bottom_area:
        print(f"{w['text']} at Y: {w['top']}")


[Y=114] Bill To : CW Management Pty Ltd B655
[Y=117] Customer No. : PH1871
[Y=135] Invoice No. : 91060
[Y=138] Ship To : Chemist Warehouse Townsville B158
[Y=150] Invoice Date : 15.08.23
[Y=159] Address : 126-128 Sturt St
[Y=165] Due Date : 30.09.23
[Y=171] TOWNSVILLE QLD 4810
[Y=183] AUSTRALIA Invoice Amount : $ 187.25 AUD
[Y=231] TTAAXX IINNVVOOIICCEE
[Y=237] PPaaggee 11 ooff 11
[Y=255] UUnniitt TToottaall TToottaall
[Y=261] NNoo.. PPeeggNNoo.. TTyyppee NNoo.. SSiizzee CCoolloouurr DDeessccrriippttiioonn QQ''ttyy UUnniitt GGSSTT
[Y=267] PPrriiccee eexxccll..GGSSTT IInnccll..GGSSTT
[Y=282] 11 DD0088 TTNN11--001122LL--CC11 33--99 NN..PPKK//OORR//YYLLLLaaddiieess'' CCaassuuaall LLooww ccuutt 11 DDzz $$4422..7788 $$4422..7788 $$44..2288 $$4477..0066
[Y=294] 22 DD1122 TTNN22--000022MM--BB11 77--1111 WWHH MMeenn''ss CCaassuuaall AAnnkklleett 11 DDzz $$4422..7788 $$4422..7788 $$44..2288 $$4477..0066
[Y=303] PPDD0033 1111--1144 MMeenn''ss BBuussiinneessss DDiiaabbeettiicc FFrriieennddllyy EE

In [24]:
import fitz  # PyMuPDF

# Define your keywords
HEADER_KEYWORDS = ["description", "qty", "q'ty", "unit", "item", "no", "colour", "code", "total"]
FOOTER_KEYWORDS = ["total", "subtotal", "gst", "amount", "grand total"]

def print_blocks(title, blocks):
    print(f"\n=== {title.upper()} ===")
    for b in blocks:
        y = b[1]
        content = b[4].strip().replace("\n", " ")
        print(f"Y: {y:.2f} | {content}")

def find_block_indexes(blocks, skip_footer_lines=2):
    header_idx = None
    footer_idx = None

    # Step 1: Find header line index
    for i, block in enumerate(blocks):
        text = block[4].replace("\n", " ").lower()
        header_hits = sum(1 for kw in HEADER_KEYWORDS if kw in text)
        if header_hits >= 2:
            header_idx = i
            print(f"✅ HEADER detected at Y={block[1]:.2f} | Line: {text}")
            break

    # Step 2: Find footer after header line
    if header_idx is not None:
        for j in range(header_idx + 1, len(blocks)):
            text = blocks[j][4].replace("\n", " ").lower()
            if any(kw in text for kw in FOOTER_KEYWORDS):
                # Ensure footer index is a few lines above actual match
                footer_idx = max(header_idx + 1, j - skip_footer_lines)
                print(f"✅ FOOTER detected at Y={blocks[j][1]:.2f} | Line: {text}")
                break

    return header_idx, footer_idx

# --- Main Logic ---
doc = fitz.open(r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf")
page = doc[0]
blocks = page.get_text("blocks")

# Sort blocks by Y-coordinate (top to bottom)
sorted_blocks = sorted(blocks, key=lambda b: b[1])

# Detect header/footer block indexes
header_idx, footer_idx = find_block_indexes(sorted_blocks)

if header_idx is None or footer_idx is None:
    raise ValueError("❌ Could not detect both header and footer lines. Adjust keyword lists or debug block content.")

# Define areas
top_area = sorted_blocks[:header_idx]
line_items_area = sorted_blocks[header_idx:footer_idx]
bottom_area = sorted_blocks[footer_idx:]

# Output areas
print_blocks("Top Area", top_area)
print_blocks("Line Items Area", line_items_area)
print_blocks("Bottom Area", bottom_area)


✅ HEADER detected at Y=259.72 | Line: no. no. pegno. pegno. type no. type no. size size colour colour description description q'ty q'ty unit unit 
✅ FOOTER detected at Y=259.72 | Line: excl.gst gst gst 

=== TOP AREA ===
Y: 113.34 | Bill  To   :   CW Management Pty Ltd B655 Customer No. : PH1871
Y: 132.42 | Ship  To : Chemist Warehouse Townsville B158 Invoice No. : 91060
Y: 148.74 | Invoice Date : 15.08.23 126-128 Sturt St TOWNSVILLE QLD 4810 AUSTRALIA
Y: 157.39 | Address : Due Date : 30.09.23
Y: 181.26 | Invoice Amount : $ 187.25 AUD
Y: 228.72 | TAX INVOICE TAX INVOICE Page Page 11 of of 11
Y: 252.87 | Unit
Y: 252.87 | Unit
Y: 252.87 | Total
Y: 252.87 | Total
Y: 252.87 | Total
Y: 252.87 | Total

=== LINE ITEMS AREA ===
Y: 259.72 | No. No. PegNo. PegNo. Type No. Type No. Size Size Colour Colour Description Description Q'ty Q'ty Unit Unit

=== BOTTOM AREA ===
Y: 259.72 | excl.GST GST GST
Y: 265.36 | Price
Y: 265.36 | Price
Y: 265.36 | excl.GST
Y: 265.36 | Incl.GST
Y: 265.36 | Incl.GST
Y

In [7]:
line_items_area

[{'text': 'TTAAXX',
  'x0': 265.91870223711,
  'x1': 289.42895464751683,
  'top': 230.82727132313926,
  'doctop': 230.82727132313926,
  'bottom': 245.19341757624534,
  'upright': True,
  'height': 14.36614625310608,
  'width': 23.51025241040685,
  'direction': 'ltr'},
 {'text': 'IINNVVOOIICCEE',
  'x0': 292.79166801184147,
  'x1': 341.3067121056883,
  'top': 230.82727132313926,
  'doctop': 230.82727132313926,
  'bottom': 245.19341757624534,
  'upright': True,
  'height': 14.36614625310608,
  'width': 48.515044093846825,
  'direction': 'ltr'},
 {'text': 'PPaaggee',
  'x0': 494.1621748842923,
  'x1': 516.4463292063314,
  'top': 237.60521835638053,
  'doctop': 237.60521835638053,
  'bottom': 247.1827311117828,
  'upright': True,
  'height': 9.577512755402267,
  'width': 22.284154322039058,
  'direction': 'ltr'},
 {'text': '11',
  'x0': 530.2789155432353,
  'x1': 535.6072529570099,
  'top': 237.60521835638053,
  'doctop': 237.60521835638053,
  'bottom': 247.1827311117828,
  'upright': True

In [5]:
print(bottom_area)

[{'text': 'SATCHEL*1', 'x0': 23.997934449153, 'x1': 68.60456494073256, 'top': 332.04346378923333, 'doctop': 332.04346378923333, 'bottom': 341.6209765446356, 'upright': True, 'height': 9.577512755402267, 'width': 44.60663049157956, 'direction': 'ltr'}, {'text': 'Total', 'x0': 320.03679165126374, 'x1': 341.3245779382675, 'top': 330.60833958647885, 'doctop': 330.60833958647885, 'bottom': 340.1858523418811, 'upright': True, 'height': 9.577512755402267, 'width': 21.28778628700377, 'direction': 'ltr'}, {'text': "Q'ty", 'x0': 344.0837509583653, 'x1': 360.7537546214555, 'top': 330.60833958647885, 'doctop': 330.60833958647885, 'bottom': 340.1858523418811, 'upright': True, 'height': 9.577512755402267, 'width': 16.670003663090256, 'direction': 'ltr'}, {'text': '3', 'x0': 399.00075075, 'x1': 404.32748755268864, 'top': 330.60833958647885, 'doctop': 330.60833958647885, 'bottom': 340.1858523418811, 'upright': True, 'height': 9.577512755402267, 'width': 5.326736802688629, 'direction': 'ltr'}, {'text':

In [11]:
pdf_path = r'C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf'

In [12]:
import fitz  # PyMuPDF

HEADER_KEYWORDS = ["description", "qty", "q'ty", "unit", "item", "pegno", "no", "colour", "code", "price"]
FOOTER_KEYWORDS = ["total", "subtotal", "gst", "grand total", "incl", "excl", "amount"]

def print_blocks(title, blocks):
    print(f"\n=== {title.upper()} ===")
    for b in blocks:
        y = b[1]
        content = b[4].strip().replace("\n", " ")
        print(f"Y: {y:.2f} | {content}")

def find_header_footer_indexes(blocks, footer_min_offset=2):
    header_idx = None
    footer_idx = None

    for i, block in enumerate(blocks):
        text = block[4].replace("\n", " ").lower()
        if header_idx is None:
            header_hits = sum(1 for kw in HEADER_KEYWORDS if kw in text)
            if header_hits >= 2:
                header_idx = i
                print(f"✅ HEADER (line items start) detected at Y={block[1]:.2f} | Line: {text}")
        elif i > header_idx + footer_min_offset:
            if any(kw in text for kw in FOOTER_KEYWORDS):
                footer_idx = i
                print(f"✅ FOOTER detected at Y={block[1]:.2f} | Line: {text}")
                break

    return header_idx, footer_idx

# --- Main Logic ---
doc = fitz.open(pdf_path)
page = doc[0]
blocks = page.get_text("blocks")

# Sort by vertical position
sorted_blocks = sorted(blocks, key=lambda b: b[1])

# Find header/footer indexes with a buffer
header_idx, footer_idx = find_header_footer_indexes(sorted_blocks)

if header_idx is None:
    raise ValueError("❌ Header not found — check keywords or invoice layout.")
if footer_idx is None:
    footer_idx = len(sorted_blocks)

# Define areas
top_area = sorted_blocks[:header_idx]
line_items_area = sorted_blocks[header_idx:footer_idx]
bottom_area = sorted_blocks[footer_idx:]

# Print
print_blocks("Top Area", top_area)
print_blocks("Line Items Area", line_items_area)
print_blocks("Bottom Area", bottom_area)


✅ HEADER (line items start) detected at Y=259.72 | Line: no. no. pegno. pegno. type no. type no. size size colour colour description description q'ty q'ty unit unit 
✅ FOOTER detected at Y=265.36 | Line: excl.gst 

=== TOP AREA ===
Y: 113.34 | Bill  To   :   CW Management Pty Ltd B655 Customer No. : PH1871
Y: 132.42 | Ship  To : Chemist Warehouse Townsville B158 Invoice No. : 91060
Y: 148.74 | Invoice Date : 15.08.23 126-128 Sturt St TOWNSVILLE QLD 4810 AUSTRALIA
Y: 157.39 | Address : Due Date : 30.09.23
Y: 181.26 | Invoice Amount : $ 187.25 AUD
Y: 228.72 | TAX INVOICE TAX INVOICE Page Page 11 of of 11
Y: 252.87 | Unit
Y: 252.87 | Unit
Y: 252.87 | Total
Y: 252.87 | Total
Y: 252.87 | Total
Y: 252.87 | Total

=== LINE ITEMS AREA ===
Y: 259.72 | No. No. PegNo. PegNo. Type No. Type No. Size Size Colour Colour Description Description Q'ty Q'ty Unit Unit
Y: 259.72 | excl.GST GST GST
Y: 265.36 | Price
Y: 265.36 | Price

=== BOTTOM AREA ===
Y: 265.36 | excl.GST
Y: 265.36 | Incl.GST
Y: 265.36 |

In [13]:
import fitz # PyMuPDF

doc = fitz.open(pdf_path)
page = doc.load_page(0) # Load the first page

# Extract text in a structured format (e.g., as a list of blocks)
text_blocks = page.get_text("blocks")

# You would then need to parse these blocks to identify and reconstruct tables

In [15]:
import camelot

tables = camelot.read_pdf(pdf_path, pages="1", flavor="stream")

if tables.n == 0:
    print("No tables found on page 1.")
else:
    print(f"Found {tables.n} tables.")
    print(tables[0].df)

Found 1 tables.
          0       1            2       3                                 4   \
0                                                                             
1                                                                             
2     No.No.  PegNo.     Type No.    Size                            Colour   
3                                                                             
4          1  D08D08  TN1-012L-C1  3-93-9  N.PK/OR/YLLadies' Casual Low cut   
5          2  D12D12  TN2-002M-B1    7-11                              WHWH   
6          3    PD03  BU4-002K-C2   11-14                              BKBK   
7          4     COU          COU                                             
8  SATCHEL*1                                                                  

                                              5         6       7       8   \
0                                    TAX INVOICE                             
1                                    

In [16]:
import camelot

tables = camelot.read_pdf(pdf_path, pages="1", flavor="stream")
df = tables[0].df

In [17]:
# Drop rows that are all empty or nearly empty
df = df[df.apply(lambda row: row.astype(str).str.strip().str.len().sum() > 5, axis=1)].reset_index(drop=True)
df

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,TAX INVOICE,,,,Page,1\nofof\n1
1,,,,,,,,,Unit,Total,Total
2,No.No.,PegNo.,Type No.,Size,Colour,Description,Q'tyQ'ty,Unit,,,GSTGST
3,,,,,,,,,Price,excl.GST,Incl.GST
4,1,D08D08,TN1-012L-C1,3-93-9,N.PK/OR/YLLadies' Casual Low cut,,1,DzDz,$42.78,$42.78,$4.28\n$47.06
5,2,D12D12,TN2-002M-B1,7-11,WHWH,Men's Casual Anklet,1,DzDz,$42.78,$42.78,$4.28\n$47.06
6,3,PD03,BU4-002K-C2,11-14,BKBK,Men's Business Diabetic Friendly Extra Large,1,DzDz,$64.17,$64.17,$6.42\n$70.59
7,4,COU,COU,,,Courier Fee,1,BoxBox,$20.50,$20.50,$2.05\n$22.55
8,SATCHEL*1,,,,,Total Q'ty,3,Dz,,$170.23\n$17.02,$187.25


In [18]:
import fitz  # PyMuPDF
import camelot

# Step 1: Extract tables
tables = camelot.read_pdf(pdf_path, pages="1", flavor="stream")

if tables.n == 0:
    print("❌ No tables found.")
    exit()

print(f"✅ Found {tables.n} table(s)")

table = tables[0]
x1, y1, x2, y2 = table._bbox  # Camelot gives (x1, y1, x2, y2) = (left, bottom, right, top)
print(f"📏 Table Bounding Box: L={x1}, B={y1}, R={x2}, T={y2}")

# Step 2: Read blocks from PyMuPDF
doc = fitz.open(pdf_path)
page = doc[0]
blocks = sorted(page.get_text("blocks"), key=lambda b: b[1])  # sort by Y (top)

# Step 3: Classify blocks into regions
top_area = []
line_items_area = []
bottom_area = []

for b in blocks:
    y_top = b[1]
    y_bottom = b[3]
    text = b[4].strip().replace("\n", " ")
    if y_bottom > y2:
        top_area.append((y_top, text))
    elif y_top < y1:
        bottom_area.append((y_top, text))
    else:
        line_items_area.append((y_top, text))

# Step 4: Print each area
def print_blocks(label, area):
    print(f"\n=== {label.upper()} ===")
    for y, text in area:
        print(f"Y: {y:.2f} | {text}")

print_blocks("Top Area", top_area)
print_blocks("Line Items Area", line_items_area)
print_blocks("Bottom Area", bottom_area)


✅ Found 1 table(s)
📏 Table Bounding Box: L=11.118363147372495, B=506.6267957996113, R=579.0975653378011, T=628.2825802854215

=== TOP AREA ===
Y: 617.13 | Electronic Funds Transfer Cheque in the Mail Credit Card Payment For Visa & Mastercard payments, please complete the following and mail or fax back to 02 9748 6211:
Y: 633.12 | Preferred Method of Payment Please make your cheque payable
Y: 647.73 | to Fashionova Pty Ltd Account ......: Fashionova Pty Ltd Bank ...........: CBA BSB/Acc No : 062 256/1042 5214
Y: 657.00 | Visa ____  Mastercard ____  Expiry:_______/_______
Y: 664.68 | (T/A SOX & LOX)
Y: 680.04 | Name on card: _______________________________
Y: 684.12 | Unit 3 / 10-11 Millennium Court Silverwater NSW 2128
Y: 684.84 | * Please quote  your Customer NO .: Or Invoice NO : for the payment.
Y: 699.36 | PH1871 91060
Y: 702.96 | Card#: ______________________________________
Y: 723.15 | Please note: 1.88% credit card surcharge applies.
Y: 827.49 | Page 1 1 of

=== LINE ITEMS AREA =

### Deep OCR

In [8]:
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

def load_and_ocr(pdf_path):
    doc = DocumentFile.from_pdf(pdf_path)
    model = ocr_predictor(pretrained=True)
    result = model(doc)
    return result

def get_word_boxes(result):
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    x1, y1 = word.geometry[0]
                    x2, y2 = word.geometry[1]
                    words.append({
                        "text": word.value,
                        "x": (x1 + x2) / 2,
                        "y": (y1 + y2) / 2,
                        "x1": x1,
                        "x2": x2,
                        "y1": y1,
                        "y2": y2,
                    })
    return words

def cluster_by_axis(words, axis='y', eps=0.01, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words

def build_table(words):
    words = cluster_by_axis(words, 'y', eps=0.01)  # row clustering
    words = cluster_by_axis(words, 'x', eps=0.02)  # column clustering

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    # Reconstruct grid
    sorted_rows = sorted(rows.items())
    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []

    for _, row in sorted_rows:
        line = [row.get(col, "") for col in all_cols]
        table_data.append(line)

    return pd.DataFrame(table_data)

# Run everything
def extract_invoice_table(pdf_path):
    ocr_result = load_and_ocr(pdf_path)
    words = get_word_boxes(ocr_result)
    table_df = build_table(words)
    return table_df

# Example
if __name__ == "__main__":
    path = r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\B035_CN-016337_20240906_V1127.pdf"  # Replace with your PDF path
    df = extract_invoice_table(path)
    print(df.to_string(index=False, header=False))


c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\doctr\models\utils\pytorch.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe

       336            
      1300            
  Integria            
       AUS            
       844            
      copy  HEALTHCARE
 CN-016337            
      B655         To:
    Street            
      3072            
       AUS            
SO-0314984            
Poari-Down            
      Nett    Material
   -105.09   EPTABP150
    -97.61     EPTRN90
   -202.70 Receivable:
       336      Phone:
         %        Fax:
       End      Email:
                Notes:
    133956            
     date:            
       1/1            


In [3]:
path = r"C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf"  # Replace with your PDF path

In [4]:
def build_table(words, min_cols=4, min_tokens=3):
    words = cluster_by_axis(words, 'y', eps=0.01)
    words = cluster_by_axis(words, 'x', eps=0.02)

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    # Get all unique columns
    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []

    for _, row in sorted(rows.items()):
        line = [row.get(col, "") for col in all_cols]
        non_empty = sum(1 for cell in line if cell.strip())
        if non_empty >= min_tokens:
            table_data.append(line)

    return pd.DataFrame(table_data)

def cluster_by_axis(words, axis='y', eps=0.008, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words


In [5]:
import numpy as np
import pandas as pd
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from sklearn.cluster import DBSCAN
import re


def is_numeric(val):
    return bool(re.match(r"^-?\d+(\.\d+)?$", val.replace(",", "")))

def cluster_by_axis(words, axis='y', eps=0.01, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words

def get_word_boxes(result):
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    x1, y1 = word.geometry[0]
                    x2, y2 = word.geometry[1]
                    words.append({
                        "text": word.value,
                        "x": (x1 + x2) / 2,
                        "y": (y1 + y2) / 2,
                        "x1": x1,
                        "x2": x2,
                        "y1": y1,
                        "y2": y2,
                    })
    return words

def build_table(words, min_cols=4, min_tokens=3):
    words = cluster_by_axis(words, 'y', eps=0.008)
    words = cluster_by_axis(words, 'x', eps=0.01)

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    # Get all unique columns
    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []

    for _, row in sorted(rows.items()):
        line = [row.get(col, "") for col in all_cols]
        non_empty = sum(1 for cell in line if cell.strip())
        if non_empty >= min_tokens:
            table_data.append(line)

    df = pd.DataFrame(table_data)

    # Drop very sparse columns
    df = df.loc[:, df.apply(lambda col: col.astype(str).str.strip().astype(bool).sum() > 2)]

    return df.reset_index(drop=True)


def extract_line_items_from_invoice(pdf_path, export_to=None):
    """
    Extracts tabular line items from an invoice using OCR and layout clustering.

    Args:
        pdf_path (str): Path to PDF invoice
        export_to (str): Optional. 'csv' or 'excel' to save the output.

    Returns:
        pd.DataFrame: Extracted line item table
    """
    # Step 1: OCR
    print(f"🔍 Running OCR on: {pdf_path}")
    doc = DocumentFile.from_pdf(pdf_path)
    model = ocr_predictor(pretrained=True)
    result = model(doc)

    # Step 2: Get word layout
    words = get_word_boxes(result)

    if not words:
        print("⚠️ No text found.")
        return pd.DataFrame()

    # Step 3: Build table using layout clustering
    df = build_table(words)

    # Step 4: Save if requested
    if export_to == "csv":
        out_path = pdf_path.replace(".pdf", "_lineitems.csv")
        df.to_csv(out_path, index=False)
        print(f"✅ Saved to: {out_path}")
    elif export_to == "excel":
        out_path = pdf_path.replace(".pdf", "_lineitems.xlsx")
        df.to_excel(out_path, index=False)
        print(f"✅ Saved to: {out_path}")

    return df

if __name__ == "__main__":
    #pdf = "invoice.pdf"
    df = extract_line_items_from_invoice(path, export_to="excel")

    print("\n📦 Final Line Item Table:")
    print(df.to_string(index=False, header=False))


🔍 Running OCR on: C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\B035_CN-016337_20240906_V1127.pdf


c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\doctr\models\utils\pytorch.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe

✅ Saved to: C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\B035_CN-016337_20240906_V1127_lineitems.xlsx

📦 Final Line Item Table:
        PTY LIMITED Customer   1300                 336                                                                                                        
70096496212         Customer   1300                                                                                                                            
       4113            Email      : orders@integria.com  Integria                                                                                              
                                                                  adjustment     note       copy                                                               
 Management     Ltd                                       Chemist      Date:          06/09/2024            Ship Elizabeth Street      B035                    
         44  Street                

In [6]:
import numpy as np
import pandas as pd
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from sklearn.cluster import DBSCAN
import re


def is_numeric(val):
    return bool(re.match(r"^-?\d+(\.\d+)?$", val.replace(",", "")))

def cluster_by_axis(words, axis='y', eps=0.01, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words

def get_word_boxes(result):
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    x1, y1 = word.geometry[0]
                    x2, y2 = word.geometry[1]
                    words.append({
                        "text": word.value,
                        "x": (x1 + x2) / 2,
                        "y": (y1 + y2) / 2,
                        "x1": x1,
                        "x2": x2,
                        "y1": y1,
                        "y2": y2,
                    })
    return words

def build_table(words, min_cols=4, min_tokens=3):
    words = cluster_by_axis(words, 'y', eps=0.008)
    words = cluster_by_axis(words, 'x', eps=0.01)

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []
    row_map = []

    for cluster_id, row in sorted(rows.items()):
        line = [row.get(col, "") for col in all_cols]
        non_empty = sum(1 for cell in line if cell.strip())
        if non_empty >= min_tokens:
            table_data.append(line)
            row_map.append(cluster_id)

    df = pd.DataFrame(table_data)
    df = df.loc[:, df.apply(lambda col: col.astype(str).str.strip().astype(bool).sum() > 2)]
    return df.reset_index(drop=True), sorted(row_map)

def extract_header_lineitems_footer(pdf_path):
    print(f"🔍 Running OCR on: {pdf_path}")
    doc = DocumentFile.from_pdf(pdf_path)
    model = ocr_predictor(pretrained=True)
    result = model(doc)

    words = get_word_boxes(result)
    if not words:
        print("⚠️ No text found.")
        return None, None, None

    df, row_ids = build_table(words)

    # Step 1: Identify candidate line item rows (at least 1 number + not sparse)
    numeric_rows = df.apply(lambda row: sum(is_numeric(str(cell)) for cell in row), axis=1)
    valid_rows = numeric_rows >= 1

    # Step 2: Find longest continuous block of valid rows
    max_block = []
    current = []
    for i, is_valid in enumerate(valid_rows):
        if is_valid:
            current.append(i)
        else:
            if len(current) > len(max_block):
                max_block = current
            current = []
    if len(current) > len(max_block):
        max_block = current

    # Step 3: Split
    if not max_block:
        print("⚠️ No line item block detected.")
        return df, None, None

    start, end = max_block[0], max_block[-1] + 1
    header = df.iloc[:start].reset_index(drop=True)
    line_items = df.iloc[start:end].reset_index(drop=True)
    footer = df.iloc[end:].reset_index(drop=True)

    return header, line_items, footer

# Example usage
if __name__ == "__main__":
    pdf_path = "invoice.pdf"
    header, line_items, footer = extract_header_lineitems_footer(path)

    print("\n🧾 Header:")
    print(header.to_string(index=False, header=False))

    print("\n📦 Line Items:")
    print(line_items.to_string(index=False, header=False))

    print("\n📑 Footer:")
    print(footer.to_string(index=False, header=False))

🔍 Running OCR on: C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Invoices\B035_CN-016337_20240906_V1127.pdf


c:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\CompleteWL\lib\site-packages\doctr\models\utils\pytorch.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe


🧾 Header:
        PTY LIMITED Customer 1300                 336                                                                                     
70096496212         Customer 1300                                                                                                         
       4113            Email    : orders@integria.com  Integria                                                                           
                                                                adjustment     note       copy                                            
 Management     Ltd                                     Chemist      Date:          06/09/2024           Ship Elizabeth Street B035       
         44  Street                                         132        No:               55327                Elizabeth                   
        VIC    3072                                   MELBOURNE       Ref:                OOD:    x5                VIC                   
                

In [13]:
import numpy as np
import pandas as pd
import pdfplumber
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from sklearn.cluster import DBSCAN
import re


def is_numeric(val):
    return bool(re.match(r"^-?\d+(\.\d+)?$", val.replace(",", "")))

def cluster_by_axis(words, axis='y', eps=0.01, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words

def get_word_boxes(result):
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    x1, y1 = word.geometry[0]
                    x2, y2 = word.geometry[1]
                    words.append({
                        "text": word.value,
                        "x": (x1 + x2) / 2,
                        "y": (y1 + y2) / 2,
                        "x1": x1,
                        "x2": x2,
                        "y1": y1,
                        "y2": y2,
                    })
    return words

def clean_ocr_text(text):
    if not isinstance(text, str): return text
    # Remove repeated characters: e.g., UUnniitt → Unit
    text = re.sub(r'(.)\1{1,}', r'\1', text)
    # Fix common OCR misreads like “..” or double quotes
    text = re.sub(r"[“”\"']+", "\"", text)
    return text.strip()

def build_table(words, min_cols=4, min_tokens=3):
    words = cluster_by_axis(words, 'y', eps=0.008)
    words = cluster_by_axis(words, 'x', eps=0.01)

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []
    row_map = []

    for cluster_id, row in sorted(rows.items()):
        line = [row.get(col, "") for col in all_cols]
        non_empty = sum(1 for cell in line if cell.strip())
        if non_empty >= min_tokens:
            table_data.append(line)
            row_map.append(cluster_id)

    df = pd.DataFrame(table_data)
    df = df.loc[:, df.apply(lambda col: col.astype(str).str.strip().astype(bool).sum() > 2)]
    return df.reset_index(drop=True), sorted(row_map)

def detect_lineitem_start(df):
    header_keywords = ["Material", "UOM", "Ordered", "Shipped", "RRP", "Unit", "Price", "Discount", "Amount"]
    for i, row in df.iterrows():
        row_text = " ".join(str(cell).lower() for cell in row if isinstance(cell, str))
        if any(keyword.lower() in row_text for keyword in header_keywords):
            return i
    return None

def extract_with_doctr(pdf_path):
    doc = DocumentFile.from_pdf(pdf_path)
    model = ocr_predictor(pretrained=True)
    result = model(doc)
    words = get_word_boxes(result)
    if not words:
        print("⚠️ No text found.")
        return None, None, None

    df, row_ids = build_table(words)
    line_start_idx = detect_lineitem_start(df)
    if line_start_idx is None:
        print("⚠️ Could not detect line item header row.")
        return df, None, None

    line_items = []
    for i in range(line_start_idx + 1, len(df)):
        if df.iloc[i].apply(lambda x: is_numeric(str(x))).sum() >= 1:
            line_items.append(i)
        else:
            break

    start = line_start_idx
    end = line_items[-1] + 1 if line_items else start + 1

    header = df.iloc[:start].reset_index(drop=True)
    line_items_df = df.iloc[start:end].reset_index(drop=True)
    footer = df.iloc[end:].reset_index(drop=True)
    return header, line_items_df, footer

def extract_with_pdfplumber(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])
                return None, df, None
    return None, None, None

def extract_invoice_items(pdf_path):
    print(f"🔍 Running extraction on: {pdf_path}")
    try:
        _, df, _ = extract_with_pdfplumber(pdf_path)
        if df is not None and len(df.columns) >= 3:
            print("✅ Extracted with pdfplumber.")
            return None, df, None
        else:
            raise ValueError("Fallback to OCR")
    except:
        print("⚠️ Falling back to Doctr OCR.")
        return extract_with_doctr(pdf_path)

# Example usage
if __name__ == "__main__":

    header, line_items, footer = extract_invoice_items(path)
    line_items = line_items.applymap(clean_ocr_text) if line_items is not None else None
    header = header.applymap(clean_ocr_text) if header is not None else None
    footer = footer.applymap(clean_ocr_text) if footer is not None else None


    print("\n🧾 Header:")
    if header is not None:
        print(header.to_string(index=False, header=False))

    print("\n📦 Line Items:")
    if line_items is not None:
        print(line_items.to_string(index=False, header=False))

    print("\n📑 Footer:")
    if footer is not None:
        print(footer.to_string(index=False, header=False))


🔍 Running extraction on: C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf
✅ Extracted with pdfplumber.

🧾 Header:

📦 Line Items:
     None None        None None None None      None None                                        None None Unit   None    None   None   None None    None
     None None        None None None None      None None                                        None None        None    None   None   None None    None
        1  D08 TN1-012L-C1 None  3-9 None N.PK/OR/Y None                      Ladies" Casual Low cut    1   Dz $42.78    None $42.78  $4.28 None  $47.06
        2  D12  TN2-02M-B1 None  7-1 None        WH None                         Men"s Casual Anklet    1   Dz $42.78    None $42.78  $4.28 None  $47.06
        3 PD03  BU4-02K-C2 None 1-14 None        BK None Men"s Busines Diabetic Friendly Extra Large    1   Dz $64.17    None $64.17  $6.42 None  $70.59
        4  COU         COU None      None

In [4]:
import numpy as np
import pandas as pd
import pdfplumber
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from sklearn.cluster import DBSCAN
import re


def is_numeric(val):
    return bool(re.match(r"^-?\d+(\.\d+)?$", val.replace(",", "")))

def clean_ocr_text(text):
    if not isinstance(text, str): return text
    text = re.sub(r'(.)\1{1,}', r'\1', text)  # Remove repeated characters
    text = re.sub(r"[“”\"']+", '"', text)
    return text.strip()

def filter_junk_rows(df):
    if df is None: return None
    df = df.reset_index(drop=True)  # 👈 ensures aligned index
    mask = df.apply(
        lambda row: row.astype(str).str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.len().sum() > 10,
        axis=1
    )
    return df[mask].reset_index(drop=True)

def cluster_by_axis(words, axis='y', eps=0.01, min_samples=1):
    coords = np.array([[w[axis]] for w in words])
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    for word, label in zip(words, clustering.labels_):
        word[f'{axis}_cluster'] = label
    return words

def get_word_boxes(result):
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    x1, y1 = word.geometry[0]
                    x2, y2 = word.geometry[1]
                    words.append({
                        "text": word.value,
                        "x": (x1 + x2) / 2,
                        "y": (y1 + y2) / 2,
                        "x1": x1,
                        "x2": x2,
                        "y1": y1,
                        "y2": y2,
                    })
    return words

def build_table(words, min_cols=4, min_tokens=3):
    words = cluster_by_axis(words, 'y', eps=0.008)
    words = cluster_by_axis(words, 'x', eps=0.01)

    rows = {}
    for w in words:
        row_id = w['y_cluster']
        col_id = w['x_cluster']
        if row_id not in rows:
            rows[row_id] = {}
        rows[row_id][col_id] = w['text']

    all_cols = sorted({col_id for r in rows.values() for col_id in r})
    table_data = []
    row_map = []

    for cluster_id, row in sorted(rows.items()):
        line = [row.get(col, "") for col in all_cols]
        non_empty = sum(1 for cell in line if cell.strip())
        if non_empty >= min_tokens:
            table_data.append(line)
            row_map.append(cluster_id)

    df = pd.DataFrame(table_data)
    df = df.loc[:, df.apply(lambda col: col.astype(str).str.strip().astype(bool).sum() > 2)]
    return df.reset_index(drop=True), sorted(row_map)

def detect_lineitem_start(df):
    header_keywords = ["Material", "UOM", "Description", "Ordered", "Shipped", "RRP", "Unit", "Price", "Discount", "Amount"]
    for i, row in df.iterrows():
        row_text = " ".join(str(cell).lower() for cell in row if isinstance(cell, str))
        if any(keyword.lower() in row_text for keyword in header_keywords):
            return i
    return None

def extract_with_doctr(pdf_path):
    doc = DocumentFile.from_pdf(pdf_path)
    model = ocr_predictor(pretrained=True)
    result = model(doc)
    words = get_word_boxes(result)
    if not words:
        print("⚠️ No text found.")
        return None, None, None

    df, row_ids = build_table(words)
    line_start_idx = detect_lineitem_start(df)
    if line_start_idx is None:
        print("⚠️ Could not detect line item header row.")
        return df, None, None

    line_items = []
    for i in range(line_start_idx + 1, len(df)):
        if df.iloc[i].apply(lambda x: is_numeric(str(x))).sum() >= 1:
            line_items.append(i)
        else:
            break

    start = line_start_idx
    end = line_items[-1] + 1 if line_items else start + 1

    header = df.iloc[:start].reset_index(drop=True)
    line_items_df = df.iloc[start:end].reset_index(drop=True)
    footer = df.iloc[end:].reset_index(drop=True)
    return header, line_items_df, footer

def extract_with_pdfplumber(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])
                return None, df, None
    return None, None, None

def extract_invoice_items(pdf_path):
    print(f"🔍 Running extraction on: {pdf_path}")
    try:
        _, df, _ = extract_with_pdfplumber(pdf_path)
        if df is not None and len(df.columns) >= 3:
            print("✅ Extracted with pdfplumber.")
            header = footer = None
            line_items = df.reset_index(drop=True)
        else:
            raise ValueError("Fallback to OCR")
    except:
        print("⚠️ Falling back to Doctr OCR.")
        header, line_items, footer = extract_with_doctr(pdf_path)

    # Clean and filter all parts
    for part in ['header', 'line_items', 'footer']:
        obj = locals().get(part)
        if obj is not None:
            obj = obj.applymap(clean_ocr_text)
            obj = filter_junk_rows(obj)
            if part == 'header': header = obj
            elif part == 'line_items': line_items = obj
            elif part == 'footer': footer = obj

    return header, line_items, footer

# Example usage
if __name__ == "__main__":
    # pdf_path = "invoice.pdf"
    header, line_items, footer = extract_invoice_items(path)

    print("\n🧾 Header:")
    if header is not None:
        print(header.to_string(index=False, header=False))

    print("\n📦 Line Items:")
    if line_items is not None:
        print(line_items.to_string(index=False, header=False))

    print("\n📑 Footer:")
    if footer is not None:
        print(footer.to_string(index=False, header=False))

🔍 Running extraction on: C:\Users\Govind\Downloads\Work_And\Self_Learning\Invoice_Extraction_demo\Fashionova\B158_91060_20230815_V1916.pdf
✅ Extracted with pdfplumber.

🧾 Header:

📦 Line Items:
     None None        None None None None      None None                                        None None Unit   None    None   None   None None    None
     None None        None None None None      None None                                        None None        None    None   None   None None    None
        1  D08 TN1-012L-C1 None  3-9 None N.PK/OR/Y None                      Ladies" Casual Low cut    1   Dz $42.78    None $42.78  $4.28 None  $47.06
        2  D12  TN2-02M-B1 None  7-1 None        WH None                         Men"s Casual Anklet    1   Dz $42.78    None $42.78  $4.28 None  $47.06
        3 PD03  BU4-02K-C2 None 1-14 None        BK None Men"s Busines Diabetic Friendly Extra Large    1   Dz $64.17    None $64.17  $6.42 None  $70.59
        4  COU         COU None      None

In [5]:
line_items

,NNoo..,PPeeggNNoo..,TTyyppee NNoo..,SSiizzee,None,CCoolloouurr,None,DDeessccrriippttiioonn,None,QQ''ttyy,,UUnniitt\nPPrriiccee,None,TToottaall\neexxccll..GGSSTT,,GGSSTT,TToottaall\nIInnccll..GGSSTT
0,None,None,None,None,None,None,None,None,None,None,Unit,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None,None,None,,None,None,None,None,None,None
2,1,D08,TN1-012L-C1,None,3-9,None,N.PK/OR/Y,None,"Ladies"" Casual Low cut",1,Dz,$42.78,None,$42.78,$4.28,None,$47.06
3,2,D12,TN2-02M-B1,None,7-1,None,WH,None,"Men""s Casual Anklet",1,Dz,$42.78,None,$42.78,$4.28,None,$47.06
4,3,PD03,BU4-02K-C2,None,1-14,None,BK,None,"Men""s Busines Diabetic Friendly Extra Large",1,Dz,$64.17,None,$64.17,$6.42,None,$70.59
5,4,COU,COU,None,,None,None,None,Courier Fe,1,Box,$20.50,None,$20.50,$2.05,None,$2.5
6,SATCHEL*1,None,None,None,None,None,None,None,None,None,None,,$170.23,None,$17.02,None,$187.25


In [6]:
# if line_items is not None and not line_items.empty:
#     line_items.columns = [clean_ocr_text(col) if col is not None else "" for col in line_items.columns]
def fix_column_alignment_and_merge(df):
    import numpy as np

    # Step 1: Clean column names
    df.columns = [clean_ocr_text(str(col)) if col else None for col in df.columns]

    # Step 2: For None-named columns, try to merge values into adjacent valid columns
    cols = df.columns.tolist()
    new_df = df.copy()

    for idx, col in enumerate(cols):
        if col is None or col.strip() == "":
            col_data = new_df.iloc[:, idx]
            if col_data.dropna().astype(str).str.strip().any():
                # Prefer to merge to the column on the left if it's named
                if idx > 0 and cols[idx - 1] is not None:
                    left_col = cols[idx - 1]
                    new_df[left_col] = new_df[left_col].combine_first(col_data)
                # Otherwise, try the right
                elif idx < len(cols) - 1 and cols[idx + 1] is not None:
                    right_col = cols[idx + 1]
                    new_df[right_col] = new_df[right_col].combine_first(col_data)
    
    # Step 3: Drop all columns where:
    # - column name is None or empty string AND
    # - all values are None or empty
    clean_cols = []
    for idx, col in enumerate(cols):
        col_data = new_df.iloc[:, idx]
        if col is not None and col.strip() != "":
            clean_cols.append(col)
        elif col_data.dropna().astype(str).str.strip().any():
            clean_cols.append(None)  # Keep it with a default name if it has data

    return new_df.loc[:, clean_cols].reset_index(drop=True)


line_items = fix_column_alignment_and_merge(line_items)

In [ ]:
line_items

,No.,PegNo.,Type No.,Size,Colour,Description,"Q""ty",Unit\nPrice,Total\nexcl.GST,GST,Total\nIncl.GST
0,None,None,None,None,None,None,Unit,None,None,None,None
1,None,None,None,None,None,None,,None,None,None,None
2,1,D08,TN1-012L-C1,3-9,N.PK/OR/Y,"Ladies"" Casual Low cut",1,$42.78,$42.78,None,$47.06
3,2,D12,TN2-02M-B1,7-1,WH,"Men""s Casual Anklet",1,$42.78,$42.78,None,$47.06
4,3,PD03,BU4-02K-C2,1-14,BK,"Men""s Busines Diabetic Friendly Extra Large",1,$64.17,$64.17,None,$70.59
5,4,COU,COU,,None,Courier Fe,1,$20.50,$20.50,None,$2.5
6,SATCHEL*1,None,None,None,None,None,None,,$17.02,None,$187.25


In [ ]:
line_items.drop(line_items[None], axis= 1)

,No.,PegNo.,Type No.,Size,None,None,None,None,None,None,...,None,Total\nexcl.GST,None,None,None,None,None,None,GST,Total\nIncl.GST
0,None,None,None,None,None,None,None,Unit,None,None,...,None,None,None,None,None,Unit,None,None,None,None
1,None,None,None,None,None,None,None,,None,None,...,None,None,None,None,None,,None,None,None,None
2,1,D08,TN1-012L-C1,3-9,3-9,N.PK/OR/Y,"Ladies"" Casual Low cut",Dz,None,$4.28,...,$4.28,$42.78,3-9,N.PK/OR/Y,"Ladies"" Casual Low cut",Dz,None,$4.28,None,$47.06
3,2,D12,TN2-02M-B1,7-1,7-1,WH,"Men""s Casual Anklet",Dz,None,$4.28,...,$4.28,$42.78,7-1,WH,"Men""s Casual Anklet",Dz,None,$4.28,None,$47.06
4,3,PD03,BU4-02K-C2,1-14,1-14,BK,"Men""s Busines Diabetic Friendly Extra Large",Dz,None,$6.42,...,$6.42,$64.17,1-14,BK,"Men""s Busines Diabetic Friendly Extra Large",Dz,None,$6.42,None,$70.59
5,4,COU,COU,,,None,Courier Fe,Box,None,$2.05,...,$2.05,$20.50,,None,Courier Fe,Box,None,$2.05,None,$2.5
6,SATCHEL*1,None,None,None,None,None,None,None,$170.23,$17.02,...,$17.02,$17.02,None,None,None,None,$170.23,$17.02,None,$187.25
